# Final Project — *Your project title here*
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Final project starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/project/Project_Starter_Notebook.ipynb)

| Team number | Members | Track | Date |
|---|---|---|---|
| *(TODO)* | *(TODO)* | Predict / See / Assist / Evaluate | *(TODO)* |

### How to use this notebook
- This is a **skeleton** for your final notebook. Its sections mirror the [report template](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/project/templates/report-template.md), so text and figures move straight into your report.
- It **runs as-is** on placeholder data (`concrete_strength.csv`), so you can see every section working before you change anything. Run it once from top to bottom first.
- Then replace the placeholder parts with your own. Cells you need to change are marked **`# TODO (team)`**; text cells you need to fill in say *(TODO)*.
- Delete placeholder text you do not need, but **keep every section heading** — graders look for them.
- **Lab 11** (Arizona bridges) is a complete worked example of this workflow. Reuse its code and say so.
- **Track add-ons:** 5.2 (See track: ResNet-18 image features), 6.6 (yes/no classification targets), and 6.7 (Assist/Evaluate: run an LLM once and cache its answers to a CSV file). 6.6 runs on the placeholder data; 5.2 and 6.7 are switched off until you set their `RUN_...` flag to `True`. They download models (about 45 MB and 1 GB). Model downloads happen on Google's computers, not on your laptop or phone data plan.
- **Slow steps:** save slow outputs (LLM answers, image features) to CSV and load them if the file exists, as 5.2 and 6.7 do. Submit the CSV files with your notebook — **graders run your notebook from the cached outputs**.

**AI-use label:** **AI-Assisted** — you may use AI tools for help; record it in your AI-use statement (Section 10). You are responsible for every line. In the Assist and Evaluate tracks, testing an AI model is part of the method (that part is **AI-Required**: record prompts, settings, and outputs).

**Before you start:** **File → Save a copy in Drive**, then share the copy with your team (**Share → Editor**). Agree on who edits the main notebook when, so you don't overwrite each other.

### Adapting this notebook to your track
| Track | Data (Section 2) | Baseline and method (Sections 4–5) | Evaluation (Section 6) |
|---|---|---|---|
| **Predict** | A CSV table — use as is. For time series, split by time (see 3.2). | Use as is; swap in your model. For a yes/no target, use the classifiers in 6.6. | MAE/RMSE/R² in units; for yes/no targets, confusion matrix and recall (6.6). |
| **See** | Unzip images and list files (5.2 and Lab 06). | Baseline: CLIP zero-shot (Lab 01) or a simple edge-filter rule. Method: ResNet-18 features + logistic regression (5.2, Lab 06). | Accuracy, confusion matrix, recall for the important class (6.6), failure images, robustness tests (Lab 13). |
| **Assist** | Your documents (e.g., `osha/` files) **and** a test-set CSV: question, expected answer, expected source. | Baseline: the LLM with no retrieval or tools. Method: your RAG or agent (Labs 08, 10), run once and cached (6.7). | The scoring helper in 6.5: correctness, citation correctness, refusals. |
| **Evaluate** | Your test-set CSV with proven answers. | Baseline: the plain condition (plain prompt, smaller model). Method: the conditions you compare, cached (6.7). | The scoring helper in 6.5, error types, agreement between two raters. |

## 0. Setup
Run this cell first. It loads libraries, sets a random seed so results repeat, and defines `get_data()`, the same helper the labs use to fetch course files.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42          # use this everywhere a function takes random_state
np.random.seed(RANDOM_STATE)
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## 1. Problem and users
*(TODO — 4–6 sentences. Copy and improve from your proposal.)*

- **Engineering problem:** *(TODO)* — e.g., *Ordering and testing concrete is slow; a designer wants a quick estimate of compressive strength for a trial mix.*
- **Our question (measurable):** *(TODO)* — e.g., *How accurately can we predict compressive strength (MPa) from mix proportions and age, on mixes the model has never seen?*
- **Users and decision:** *(TODO)* — who uses the answer and what they decide with it.
- **What they do today without AI:** *(TODO)*
- **Success criterion:** *(TODO)* — e.g., *MAE below 6 MPa and clearly better than the baseline.*

## 2. Data and its limits
### 2.1 Load the data
Course files load with `get_data()`. For an outside dataset, either read it straight from its URL with `pd.read_csv("https://...")`, or upload the file to Colab (folder icon on the left → upload) and read it by name. Record where and when you downloaded it in the data card (2.2).

In [ ]:
# TODO (team): change DATA_FILE to your dataset (course file name, or a URL / uploaded file name).
DATA_FILE = "concrete_strength.csv"     # placeholder data
df = pd.read_csv(get_data(DATA_FILE))   # for a URL or uploaded file: pd.read_csv("your_file.csv")

print("Rows, columns:", df.shape)
df.head()

### 2.2 Data card
*(TODO — replace the placeholder entries with your own data.)*

| Item | Our data |
|---|---|
| Name and provider | *Concrete Compressive Strength* (I‑C. Yeh, 1998), via the UCI Machine Learning Repository — course file `concrete_strength.csv` |
| URL and download date | https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength · *(date)* |
| License | CC BY 4.0 |
| Size | 1,030 rows × 9 columns |
| One row is… | one concrete mix tested at one age |
| Inputs (units) | 7 ingredient amounts (kg/m³), age (days) |
| Target (units) | compressive strength (MPa) |
| How the target was measured / labeled | laboratory compression tests |
| Known limits | *(see 2.6)* |

### 2.3 A quick health check
Count missing values and look at the range of every column. Ranges catch unit problems early (a strength of 4,000 is probably psi, not MPa).

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
df.describe().T[["count", "mean", "min", "max"]].round(2)

### 2.4 Cleaning — and write down what you did
Every change to the data is recorded with counts before and after, so a reader can repeat it. Here the placeholder data has some exact duplicate rows.

In [ ]:
# TODO (team): add your own cleaning steps (missing values, unit conversions, impossible values).
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df)} exact duplicate rows: {n_before} -> {len(df)} rows")

### 2.5 Explore
Two or three plots that tell you something important. Every plot needs a title and axis labels **with units**.

In [ ]:
# TODO (team): set your target column and its units, and pick one input to plot against it.
TARGET = "strength_mpa"
TARGET_UNITS = "MPa"
X_TO_PLOT = "age_days"

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df[TARGET], bins=30, color="#1E5288", edgecolor="white")
axes[0].set_title("Distribution of the target")
axes[0].set_xlabel(f"Compressive strength ({TARGET_UNITS})")
axes[0].set_ylabel("Number of rows")

axes[1].scatter(df[X_TO_PLOT], df[TARGET], s=10, alpha=0.5, color="#AB0520")
axes[1].set_xscale("log")
axes[1].set_title("Strength rises with age, then levels off")
axes[1].set_xlabel("Age at testing (days, log scale)")
axes[1].set_ylabel(f"Compressive strength ({TARGET_UNITS})")
plt.tight_layout()
plt.show()

### 2.6 Limits of the data
*(TODO — at least three specific limits. Placeholder examples for the concrete data:)*
- Lab specimens, not field concrete: no information on curing temperature, placement, or consolidation.
- Ingredient ranges are limited (cement 102–540 kg/m³); predictions outside them are guesses.
- Many mixes are tested at several ages, so rows are **not independent** — this affects how we split (3.2).
- *(Who or what is missing from the data? Which regions, climates, building types, years?)*

## 3. Method
### 3.1 Target and features
**Features** (X) are the inputs; the **target** (y) is what we predict. Engineering knowledge makes good features: the water–cement ratio is one of the strongest predictors of concrete strength.

In [ ]:
# TODO (team): add your own engineered features and list the input columns you will use.
df["wc_ratio"] = df["water_kg_m3"] / df["cement_kg_m3"]          # water-cement ratio (-)

FEATURES = ["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "superplasticizer_kg_m3",
            "coarse_agg_kg_m3", "fine_agg_kg_m3", "age_days", "wc_ratio"]
print(len(FEATURES), "features; target =", TARGET, f"({TARGET_UNITS})")

### 3.2 Split the data so the test is truly unseen
The test set must contain cases the model has **never** seen. How you split depends on the data:
- **Rows independent?** A random split is fine (`train_test_split`).
- **Same object appears in several rows** (one mix tested at several ages, one building shape at several orientations)? Split by **group**, so a whole mix is either in training or in testing.
- **Time series** (traffic, energy, flow)? Split by **time**: train on earlier data, test on later, e.g. `train = df[df["date_time"] < "2018-01-01"]`.

The placeholder uses a group split: rows with identical ingredient amounts are the same mix.

In [ ]:
# TODO (team): choose your split. Set GROUP_COLUMNS = None for a plain random split.
GROUP_COLUMNS = ["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "superplasticizer_kg_m3",
                 "coarse_agg_kg_m3", "fine_agg_kg_m3"]          # same ingredients = same mix

if GROUP_COLUMNS:
    groups = df[GROUP_COLUMNS].astype(str).agg("|".join, axis=1)   # one text label per mix
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(df, groups=groups))
    train, test = df.iloc[train_idx], df.iloc[test_idx]
    print("Distinct groups:", groups.nunique())
else:
    train, test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"Training rows: {len(train)}   Test rows: {len(test)}")

## 4. Baseline
A **baseline** is the simplest reasonable approach. Without one, a number like "MAE = 5 MPa" means nothing. Here the baseline always predicts the average strength of the training set.

- Classification (yes/no target): `DummyClassifier(strategy="most_frequent")` — always the majority class. Section 6.6 runs it.
- Time series: "same hour last week" or "same as the previous hour."
- Assist / Evaluate: the LLM with no retrieval or tools, or the plain prompt.

In [ ]:
def regression_metrics(name, y_true, y_pred):
    """Return a dictionary of test metrics in the target's units."""
    return {"model": name,
            f"MAE ({TARGET_UNITS})": mean_absolute_error(y_true, y_pred),
            f"RMSE ({TARGET_UNITS})": mean_squared_error(y_true, y_pred) ** 0.5,
            "R²": r2_score(y_true, y_pred)}

results = []     # one row per model; becomes the results table in Section 6

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
results.append(regression_metrics("Baseline: predict the training mean", y_test, baseline.predict(X_test)))
results[-1]

## 5. Main method
*(TODO — one paragraph: what method, why it fits, key settings.)* The placeholder trains two models: linear regression (with feature scaling) and a random forest.

In [ ]:
# TODO (team): replace or add your own models. Keep random_state=RANDOM_STATE for repeatable results.
models = {
    "Linear regression (scaled)": make_pipeline(StandardScaler(), LinearRegression()),
    "Random forest (200 trees)": RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                                       random_state=RANDOM_STATE, n_jobs=-1),
}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)                 # learn from the training set only
    predictions[name] = model.predict(X_test)   # predict cases the model has never seen
    results.append(regression_metrics(name, y_test, predictions[name]))
print("Trained:", list(models))

### 5.2 See track (optional): image features with ResNet-18
This is the transfer-learning recipe from Lab 06: a ResNet-18 network trained on ImageNet turns each photo into 512 numbers (**features**), and a small logistic regression learns crack / no crack from them. Feature extraction is the slow step, so the cell **saves the features to a CSV file** and loads that file on later runs.

`stratify=labels` in the split keeps the same share of crack and no-crack photos in the training and test sets.

In [ ]:
def resnet18_features(paths, batch_size=32):
    """Return an array of shape (n_photos, 512): ResNet-18 features with the last layer removed, as in Lab 06."""
    import torch
    from PIL import Image
    from torchvision.models import resnet18, ResNet18_Weights
    weights = ResNet18_Weights.DEFAULT              # ImageNet weights: about 45 MB the first time
    net = resnet18(weights=weights).eval()
    net.fc = torch.nn.Identity()                    # remove the 1,000-class layer -> 512 features per photo
    prepare = weights.transforms()                  # the same photo preparation used in training
    feats = []
    with torch.no_grad():                           # no training here, so no gradients needed
        for start in range(0, len(paths), batch_size):
            batch = torch.stack([prepare(Image.open(p).convert("RGB")) for p in paths[start:start + batch_size]])
            feats.append(net(batch).numpy())
    return np.concatenate(feats)

# TODO (See track): set to True, and point the file lists at YOUR images.
RUN_SEE_TRACK = False
FEATURE_CACHE = "image_features.csv"                # submit this file with your notebook

if RUN_SEE_TRACK:
    import glob, os, zipfile
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import recall_score
    zipfile.ZipFile(get_data("concrete_cracks_sample.zip")).extractall(".")      # placeholder images
    crack = sorted(glob.glob("concrete_cracks/crack/*.jpg"))[:100]
    intact = sorted(glob.glob("concrete_cracks/no_crack/*.jpg"))[:100]
    paths = crack + intact
    labels = np.array([1] * len(crack) + [0] * len(intact))                     # 1 = crack, 0 = no crack

    if os.path.exists(FEATURE_CACHE) and pd.read_csv(FEATURE_CACHE)["path"].tolist() == paths:
        img_features = pd.read_csv(FEATURE_CACHE).drop(columns="path").to_numpy()
        print(f"Loaded cached features from {FEATURE_CACHE}")
    else:
        img_features = resnet18_features(paths)
        pd.DataFrame(img_features).assign(path=paths).to_csv(FEATURE_CACHE, index=False)
        print(f"Saved features for {len(paths)} photos to {FEATURE_CACHE}")

    F_tr, F_te, l_tr, l_te = train_test_split(img_features, labels, test_size=0.25, stratify=labels,
                                              random_state=RANDOM_STATE)
    head = LogisticRegression(max_iter=1000).fit(F_tr, l_tr)
    pred_img = head.predict(F_te)
    print(f"Test photos: {len(l_te)}   accuracy: {(pred_img == l_te).mean():.1%}   "
          f"recall for cracks: {recall_score(l_te, pred_img, pos_label=1):.1%}")
else:
    print("See-track cell skipped. Set RUN_SEE_TRACK = True to run it.")

## 6. Evaluation
### 6.1 Results table (test data)

In [ ]:
results_table = pd.DataFrame(results).set_index("model").round(2)
results_table

### 6.2 Predicted vs actual
Points on the dashed line are perfect predictions. Look for patterns: are high strengths under-predicted?

In [ ]:
best_name = results_table[f"MAE ({TARGET_UNITS})"].drop(results_table.index[0]).idxmin()   # skip the baseline row
best_pred = predictions[best_name]
lims = [0, max(y_test.max(), best_pred.max()) * 1.05]

plt.figure(figsize=(5, 5))
plt.scatter(y_test, best_pred, s=12, alpha=0.6, color="#1E5288")
plt.plot(lims, lims, "--", color="gray", label="perfect prediction")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel(f"Measured strength ({TARGET_UNITS})")
plt.ylabel(f"Predicted strength ({TARGET_UNITS})")
plt.title(f"Test set: {best_name}")
plt.legend()
plt.show()

### 6.3 Where does it fail? Error by subgroup
A single average hides where the model is weak. Break the error down by a meaningful group (age, material, building type, region).

In [ ]:
# TODO (team): choose a column and bins (or a category column) that make sense for your problem.
SUBGROUP_COLUMN = "age_days"
BINS = [0, 7, 28, 90, 365]          # days

errors = test.assign(predicted=np.round(best_pred, 1), abs_error=np.abs(y_test.to_numpy() - best_pred))
by_group = (errors.groupby(pd.cut(errors[SUBGROUP_COLUMN], BINS), observed=True)["abs_error"]
            .agg(test_rows="count", MAE="mean").round(2))
by_group.columns = ["test rows", f"MAE ({TARGET_UNITS})"]
by_group

### 6.4 The worst cases
Look at real failures, not just averages. What do the worst predictions have in common?

In [ ]:
# TODO (team): choose the columns that help explain your worst cases.
SHOW_COLUMNS = ["cement_kg_m3", "water_kg_m3", "age_days", "wc_ratio"]
errors.sort_values("abs_error", ascending=False).head(5)[SHOW_COLUMNS + [TARGET, "predicted", "abs_error"]].round(2)

### 6.5 Scoring helper for Assist and Evaluate projects
If your "model" is an LLM or another AI tool, your test set is a table of questions with known answers. Decide the scoring rule **before** you collect answers. The helper below pulls the number that comes right before the expected units and marks it correct if it is within 2%.

The four answers below are **made-up examples** to show how scoring works (the correct moment for a 6 m span with 12 kN/m is M = wL²/8 = 54 kN·m, with w = load in kN/m).

In [ ]:
def extract_number(text, units):
    """Return the number written just before `units` in `text` (or None if there is none)."""
    match = re.search(r"(-?\d+(?:\.\d+)?)\s*" + re.escape(units), text)
    return float(match.group(1)) if match else None

def score_numeric(text, expected, units, rel_tol=0.02):
    """True if the answer states a value within rel_tol of `expected`, in the expected units."""
    value = extract_number(text, units)
    return value is not None and abs(value - expected) <= rel_tol * abs(expected)

# Made-up example answers (replace with your tool's real outputs, saved to a CSV)
example = pd.DataFrame({
    "answer": ["M = wL²/8 = 12 × 6² / 8 = 54 kN·m",
               "M = wL²/8 = 12 × 6² / 8 = 72 kN·m",
               "The maximum moment is 0.054 MN·m",
               "The moment depends on the support conditions."],
    "expected": [54.0] * 4,
})
example["correct"] = [score_numeric(a, e, "kN·m") for a, e in zip(example["answer"], example["expected"])]
print("Correct rate:", f"{example['correct'].mean():.0%}")
example

Notice the third answer: 0.054 MN·m is physically the same moment, but the rule marks it wrong because it is not in kN·m. Is that what you want? There is no single right rule — but you must **choose it before testing**, write it down, and apply it the same way to every condition.

**Two raters:** for answers that need judgment (does the citation support the claim?), have two teammates score the same items independently and report how often they agree, e.g. `(scores["rater_1"] == scores["rater_2"]).mean()`.

### 6.6 Yes/no targets: classification (runs on the placeholder data)
Many projects predict a yes/no answer: *is this deck in low condition? is there a crack? will the wash run?* Use **0/1 labels** with **1 = the class that matters** (the problem you want to catch). Then report, on the test set:
- **recall** for class 1 — of the real problems, how many the model caught;
- **precision** for class 1 — of the model's alarms, how many were real;
- a **confusion matrix** — which mistakes happen.

Accuracy alone misleads when class 1 is rare: the baseline below never predicts class 1, so its recall is 0 even though its accuracy looks fine. Here the placeholder target is "strength below 25 MPa." `class_weight="balanced"` tells a model to pay more attention to the rarer class.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, ConfusionMatrixDisplay

# TODO (team): define YOUR 0/1 target (1 = the class that matters), e.g. (deck_condition <= 5).astype(int)
LOW_STRENGTH_MPA = 25
y_train_c = (y_train < LOW_STRENGTH_MPA).astype(int)      # 1 = low strength
y_test_c = (y_test < LOW_STRENGTH_MPA).astype(int)
print(f"Share of class 1: training {y_train_c.mean():.0%}, test {y_test_c.mean():.0%}")

classifiers = {
    "Baseline: always the majority class": DummyClassifier(strategy="most_frequent"),
    "Logistic regression (balanced)": make_pipeline(StandardScaler(),
                                                    LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest (balanced)": RandomForestClassifier(n_estimators=200, min_samples_leaf=2, class_weight="balanced",
                                                       random_state=RANDOM_STATE, n_jobs=-1),
}
class_preds, rows = {}, []
for name, clf in classifiers.items():
    pred = clf.fit(X_train, y_train_c).predict(X_test)
    class_preds[name] = pred
    rows.append({"model": name, "accuracy": accuracy_score(y_test_c, pred),
                 "recall (class 1)": recall_score(y_test_c, pred, pos_label=1, zero_division=0),
                 "precision (class 1)": precision_score(y_test_c, pred, pos_label=1, zero_division=0)})
pd.DataFrame(rows).set_index("model").round(2)

In [ ]:
# TODO (team): pick the model to show, and name your two classes.
SHOW = "Random forest (balanced)"
ConfusionMatrixDisplay.from_predictions(y_test_c, class_preds[SHOW],
                                        display_labels=["0: 25 MPa or more", "1: below 25 MPa"])
plt.title(f"Test set: {SHOW}")
plt.show()

### 6.7 Assist / Evaluate (optional): run an LLM once and cache its answers
Running an LLM on a whole test set is slow on a CPU (often 10–40 minutes). So: run each condition **once**, save every answer to a CSV file with the model name, settings, and date, and from then on **load the CSV instead of re-running the model**. Submit the CSV with your notebook — graders run your notebook from it.

The three placeholder questions have answers you can prove: M = wL²/8 = 12 × 6² / 8 = 54 kN·m (w = load, kN/m); 4,000 psi × 0.006895 MPa/psi = 27.6 MPa; heat loss Q = U·A·ΔT = 0.5 × 20 × 25 = 250 W. If you change the test set or the prompts, delete the CSV so the model runs again.

**What to expect:** when we ran these three questions through Qwen2.5-0.5B-Instruct (Nov 2026), it scored **0 of 6** under both prompts — a wrong beam method, 4,000 psi “converted” to 0.584 MPa, and answers that ran out of tokens before giving a number. A score of 0 can be a real finding, but first read the answers: make sure `max_new_tokens` is long enough and your scoring rule is not missing correct answers.

In [ ]:
# TODO (Assist/Evaluate): load YOUR test set, e.g. test_set = pd.read_csv("test_set.csv"), and set RUN_LLM = True.
RUN_LLM = False
CACHE_FILE = "llm_outputs.csv"                    # submit this file with your notebook
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"         # about 1 GB the first time

test_set = pd.DataFrame({
    "id": [1, 2, 3],
    "question": ["A simply supported beam spans 6 m and carries a uniform load of 12 kN/m. "
                 "What is the maximum bending moment in kN·m?",
                 "Convert a concrete strength of 4,000 psi to MPa.",
                 "A wall has U = 0.5 W/(m²·K), area 20 m², and a 25 K indoor-outdoor temperature difference. "
                 "What is the heat loss in W?"],
    "expected": [54.0, 27.6, 250.0],
    "units": ["kN·m", "MPa", "W"],
})
CONDITIONS = {      # condition name -> system prompt; the first one is the baseline
    "plain": "You are a helpful assistant.",
    "structured": "You are a careful engineering assistant. State the formula, show units at every step, check "
                  "the magnitude, and end with one line: ANSWER: <number> <units>.",
}

In [ ]:
import os

if os.path.exists(CACHE_FILE):
    outputs = pd.read_csv(CACHE_FILE)
    print(f"Loaded {len(outputs)} cached answers from {CACHE_FILE} - the model was NOT re-run.")
elif RUN_LLM:
    from transformers import pipeline
    chat = pipeline("text-generation", model=MODEL_NAME)
    rows = []
    for condition, system_prompt in CONDITIONS.items():
        for _, item in test_set.iterrows():
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": item["question"]}]
            reply = chat(messages, max_new_tokens=400, do_sample=False)[0]["generated_text"][-1]["content"]
            rows.append({"id": item["id"], "condition": condition, "model": MODEL_NAME, "do_sample": False,
                         "date": str(pd.Timestamp.today().date()), "answer": reply})
    outputs = pd.DataFrame(rows)
    outputs.to_csv(CACHE_FILE, index=False)
    print(f"Saved {len(outputs)} answers to {CACHE_FILE}. Submit this file with your notebook.")
else:
    outputs = None
    print("No cached answers and RUN_LLM = False - skipped. Set RUN_LLM = True to run the model once.")

if outputs is not None:
    scored = outputs.merge(test_set, on="id")
    scored["correct"] = [score_numeric(a, e, u) for a, e, u in zip(scored["answer"], scored["expected"], scored["units"])]
    print(scored.groupby("condition")["correct"].mean().rename("correct rate").round(2))
    with pd.option_context("display.max_colwidth", 500):     # show whole answers, not the first 50 characters
        display(scored[["id", "condition", "correct", "answer"]])

## 7. Verification — the five checks
Apply the course's five checks to **your own results**. Fill in the table, then run the check cells below and add your own.

| Check | What we did | What we found |
|---|---|---|
| **1. Restate** | *(TODO)* | *(TODO)* |
| **2. Units & magnitude** | *(TODO)* | *(TODO)* |
| **3. Independent check** | *(TODO)* | *(TODO)* |
| **4. Sources** | *(TODO)* | *(TODO)* |
| **5. Test & document** | *(TODO)* | *(TODO)* |

**1. Restate.** *(TODO — In two sentences in your own words: what did the model predict, for whom, and how good was it? Does that answer the question in Section 1?)*

**2. Units & magnitude.** Are predictions physically plausible? Normal-weight structural concrete is roughly 10–100 MPa; a prediction of −5 or 400 MPa means a bug or a unit mix-up.

In [ ]:
# TODO (team): set a plausible range for YOUR target, with units.
PLAUSIBLE_MIN, PLAUSIBLE_MAX = 0, 120          # MPa
print(f"Predictions range from {best_pred.min():.1f} to {best_pred.max():.1f} {TARGET_UNITS}")
assert PLAUSIBLE_MIN <= best_pred.min() and best_pred.max() <= PLAUSIBLE_MAX, "Implausible predictions: check units!"
print("Magnitude check passed.")

**3. Independent check.** Compare with something you know independently of the model: a hand calculation, a textbook trend, or a known case. Here: for the same mix, strength should rise with age, and a lower water–cement ratio should give higher strength.

In [ ]:
# TODO (team): replace with an independent check that fits your project.
typical = X_train.median()                     # a "typical" mix from the training data
typical["wc_ratio"] = typical["water_kg_m3"] / typical["cement_kg_m3"]   # keep w/c consistent with water and cement
cases = pd.DataFrame([typical] * 3)
cases["age_days"] = [7, 28, 90]
model = models[best_name]
by_age = model.predict(cases[FEATURES])
print("Predicted strength at 7, 28, 90 days:", np.round(by_age, 1), TARGET_UNITS)

lower_wc = typical.copy()
lower_wc["water_kg_m3"] *= 0.85                # 15% less water, same cement
lower_wc["wc_ratio"] = lower_wc["water_kg_m3"] / lower_wc["cement_kg_m3"]
pair = model.predict(pd.DataFrame([typical, lower_wc])[FEATURES])
print(f"w/c {typical['wc_ratio']:.2f} -> {pair[0]:.1f} {TARGET_UNITS};  w/c {lower_wc['wc_ratio']:.2f} -> {pair[1]:.1f} {TARGET_UNITS}")

print("Strength rises with age:", bool(by_age[0] < by_age[1] < by_age[2]))
print("Lower w/c gives higher strength:", bool(pair[1] > pair[0]))

**4. Sources.** *(TODO — tick each one.)*
- [ ] Every dataset has a working URL, provider, and license in the data card.
- [ ] Every formula or rule of thumb we used is cited (textbook, code section, lab).
- [ ] Every citation an AI tool gave us was checked: it exists and says what we claim.
- [ ] Code adapted from the labs or elsewhere is credited in a comment.

**5. Test & document.** `assert` statements turn checks into tests that run every time. Add tests for anything that would silently break your results.

In [ ]:
# TODO (team): keep the tests that apply and add your own.
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "Missing values in the features"
assert len(set(train.index) & set(test.index)) == 0, "A row is in both training and test sets"
if GROUP_COLUMNS:
    assert not set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]), "A group (mix) is in both sets"
assert len(best_pred) == len(y_test), "One prediction per test row"
assert np.isfinite(results_table.to_numpy(dtype=float)).all(), "A metric is not a number"
print("All tests passed.")

## 8. Responsible-AI analysis
### 8.1 Risks and failure modes
*(TODO — at least three rows. One placeholder row shown.)*

| Failure mode | Who is affected | How bad? | How would anyone notice? | What reduces the risk |
|---|---|---|---|---|
| Over-predicts strength for an unusual mix | Building occupants; the engineer of record | Serious if used for design without testing | Only by lab testing the mix | Use as a screening estimate only; always test cylinders |
| *(TODO)* | | | | |
| *(TODO)* | | | | |

### 8.2 Distribution shift: is a new case inside the training data?
A model is only trustworthy inside the range of data it learned from. This helper flags inputs outside the training range — the example mix has more cement than any mix in the data.

In [ ]:
train_range = X_train.agg(["min", "max"]).T

def outside_training_range(case):
    """List the features of `case` (a dict or Series) that fall outside the training range."""
    return [f"{f} = {case[f]:g} (trained on {train_range.loc[f, 'min']:g} to {train_range.loc[f, 'max']:g})"
            for f in FEATURES if not train_range.loc[f, "min"] <= case[f] <= train_range.loc[f, "max"]]

# TODO (team): try a case your users might really ask about.
new_mix = dict(typical)
new_mix.update(cement_kg_m3=600.0)
new_mix["wc_ratio"] = new_mix["water_kg_m3"] / new_mix["cement_kg_m3"]
flags = outside_training_range(new_mix)
print("WARNING: extrapolating ->" if flags else "Inside the training range.", flags)

### 8.3 Bias, representativeness, and accountability
*(TODO — a short paragraph for each.)*
- **Bias and representativeness:** whose infrastructure, buildings, or situations are missing or under-represented? Did error differ by subgroup (6.3)?
- **Other concerns:** privacy, security, copyright and licensing, environmental cost — whichever apply.
- **Accountability:** who is responsible if someone relies on this result? (The licensed engineer in responsible charge — never the tool.) What human review is required before it is used?

### 8.4 Model card (short)
| Item | Our model or tool |
|---|---|
| Intended use | *(TODO)* |
| Not intended for | *(TODO — e.g., final design, safety decisions, mixes outside the training range)* |
| Data | *(TODO)* |
| Performance (test set) | *(TODO — numbers with units, vs baseline)* |
| Known limits and failure modes | *(TODO)* |
| Who is accountable | *(TODO)* |

## 9. Conclusions and recommendation
*(TODO — 4–6 sentences.)*
- Answer the question from Section 1 directly, with numbers and units.
- Recommendation for the user: **use it / use only with conditions / do not use it** — and why.
- What you would do next with more time or data.

## 10. AI-use statement
Submit the full statement with the [AI-use statement template](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/project/templates/ai-use-statement.md). Summarize it here too:

*(TODO — one sentence per use, in the course form:)* "I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work." — or "We did not use AI tools for this assignment."

## 11. Reproducibility and submission
1. **Runtime → Restart session and run all.** Every cell must run without errors (aim for under 10 minutes).
2. Check that every number in your report matches the outputs here.
3. **File → Download → Download .ipynb**, name it `Team<number>_Notebook.ipynb`, and upload it to Brightspace with your report — together with any cached CSV files (for example `llm_outputs.csv`, `image_features.csv`). Due: the Week 16 deadline posted on Brightspace (Arizona time).

In [ ]:
import sys, sklearn, matplotlib
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)